# Differentiability: KHI linear eigenmode initialization in 2D.

In [ ]:
from autocvd import autocvd
autocvd(num_gpus=1)
# ruff: noqa: E402
# =======================

# general
from pathlib import Path

# jax
import jax
import jax.numpy as jnp

# plotting
import matplotlib.pyplot as plt

# astronomix constants
from astronomix import (
    FINITE_DIFFERENCE,
    BACKWARDS,
    PERIODIC_BOUNDARY,
    OPEN_BOUNDARY,
)

# astronomix containers
from astronomix import (
    SimulationConfig,
    SimulationParams,
    BoundarySettings,
    BoundarySettings1D,
)

# astronomix functions
from astronomix import (
    time_integration,
    get_registered_variables,
    get_helper_data,
    construct_primitive_state,
    finalize_config,
)

# astronomix internals for eigenmode construction
from astronomix._fluid_equations._equations import (
    conserved_state_from_primitive,
    primitive_state_from_conserved,
)
from astronomix.analysis_helpers.jacobians import single_xmode_rhs_jacobian2D

In [ ]:
figures_dir = Path("figures")

figures_dir.mkdir(exist_ok=True)

configure a small differentiable 2D Kelvin-Helmholtz simulation (reverse-mode)

In [ ]:
box_size = 1.0

num_cells = 128

gamma = 5 / 3

y_center = 0.5

background_density = 1.0

pressure = 1.0

density_contrast = 2.0

mach_number = 0.5

wavelength = box_size / 2

smoothing_length = wavelength / 10

config = SimulationConfig(
    solver_mode = FINITE_DIFFERENCE,
    progress_bar = False,
    dimensionality = 2,
    box_size = box_size,
    num_cells = num_cells,
    differentiation_mode = BACKWARDS,
    boundary_settings = BoundarySettings(
        x = BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY),   # flow direction
        y = BoundarySettings1D(OPEN_BOUNDARY, OPEN_BOUNDARY),           # transverse
    ),
)

registered_variables = get_registered_variables(config)

helper_data = get_helper_data(config)

a Kelvin-Helmholtz growth time sets a physical end time for the run

In [ ]:
c_background = float(jnp.sqrt(gamma * pressure / background_density))

v_shear = mach_number * c_background

stream_density = density_contrast * background_density

Delta = (stream_density + background_density) ** 2 / (stream_density * background_density)

t_kh = float(jnp.sqrt(Delta) * wavelength / v_shear)

params = SimulationParams(t_end = 1.5 * t_kh, C_cfl = 1.5, gamma = gamma)

build the smoothed single-interface base state (depends on y only)

In [ ]:
def single_interface(f_l, f_u, Y):
    return 0.5 * (
        f_l * (1 - jnp.tanh((Y - y_center) / smoothing_length))
        + f_u * (1 + jnp.tanh((Y - y_center) / smoothing_length))
    )

X = helper_data.geometric_centers[:, :, 0]

Y = helper_data.geometric_centers[:, :, 1]

density = single_interface(stream_density, background_density, Y)

v_x = single_interface(-v_shear / 2, v_shear / 2, Y)

v_y = jnp.zeros_like(v_x)

p = jnp.full_like(v_x, pressure)

base_state = construct_primitive_state(
    config = config,
    registered_variables = registered_variables,
    density = density,
    velocity_x = v_x,
    velocity_y = v_y,
    gas_pressure = p,
)

config = finalize_config(config, base_state.shape)

assemble the +kx Fourier-block tangent operator of the simulator and diagonalize it

In [ ]:
J = single_xmode_rhs_jacobian2D(
    base_state,
    config,
    params,
    registered_variables,
    helper_data,
    wavelength=wavelength,
)

eigvals, eigvecs = jnp.linalg.eig(J)

rho_i = registered_variables.density_index

momy_i = registered_variables.velocity_index.y

nvar, Nx, Ny = base_state.shape

modes = eigvecs.T.reshape((-1, nvar, Ny))     # [mode, variable, y]

rho0 = base_state[rho_i, 0, :]

v0 = base_state[momy_i, 0, :]

dv = modes[:, momy_i, :] / rho0 - v0 * modes[:, rho_i, :] / rho0

select the fastest-growing mode that is localized at the shear interface (the KHI mode)

In [ ]:
m = int(round(box_size / wavelength))

kx = 2.0 * jnp.pi * m / box_size

y = Y[0, :]

envelope = jnp.exp(-kx * jnp.abs(y - y_center))

localization = jnp.sum(jnp.abs(dv) ** 2 * envelope, axis=1) / jnp.sum(jnp.abs(dv) ** 2, axis=1)

growing = jnp.real(eigvals) > 0.0

score = jnp.where(growing & (localization > 0.2), jnp.real(eigvals), -jnp.inf)

idx = int(jnp.argmax(score))

lam = eigvals[idx]

print(f"selected KHI eigenmode: lambda = {float(jnp.real(lam)):.4e} + {float(jnp.imag(lam)):.4e}i")

phase-align and normalize the eigenmode so its peak transverse velocity is unity

In [ ]:
qhat = modes[idx]

dv_b = qhat[momy_i] / rho0 - v0 * qhat[rho_i] / rho0

anchor = int(jnp.argmax(jnp.abs(dv_b) * envelope))

qhat = qhat * jnp.exp(-1j * jnp.angle(dv_b[anchor]))

dv_b = qhat[momy_i] / rho0 - v0 * qhat[rho_i] / rho0

phase = jnp.exp(1j * kx * X)

max_vy = jnp.max(jnp.abs(jnp.real(dv_b[None, :] * phase)))

qhat_unit = qhat / max_vy

initialize the base state with the eigenmode at a given amplitude

In [ ]:
cons0 = conserved_state_from_primitive(base_state, gamma, config, registered_variables)

def perturbed_state(amplitude):
    delta_cons = amplitude * jnp.real(qhat_unit[:, None, :] * phase[None, :, :])
    return primitive_state_from_conserved(
        cons0 + delta_cons.astype(cons0.dtype),
        gamma,
        config,
        registered_variables,
    )

differentiate the final transverse-velocity energy w.r.t. the eigenmode amplitude

In [ ]:
def loss_fn(amplitude):
    final_state = time_integration(
        perturbed_state(amplitude),
        config,
        params,
        registered_variables,
    )
    return jnp.mean(final_state[momy_i] ** 2)

amplitude0 = v_shear / 20

loss, grad = jax.value_and_grad(loss_fn)(amplitude0)

print(f"loss = {float(loss):.4e}, d(loss)/d(amplitude) = {float(grad):.4e}")

plot the initial eigenmode perturbation and the developed KHI

In [ ]:
initial_state = perturbed_state(amplitude0)

final_state = time_integration(initial_state, config, params, registered_variables)

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].imshow(initial_state[momy_i].T, origin="lower", cmap="RdBu_r")

axs[0].set_title(r"initial eigenmode $v_y$")

axs[1].imshow(final_state[rho_i].T, origin="lower", cmap="viridis")

axs[1].set_title(f"density at $t = 1.5\\,\\tau_{{KH}}$")

for ax in axs:
    ax.set_axis_off()

fig.savefig(figures_dir / "eigen_initialization.png", dpi=200, bbox_inches="tight")